In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# Load .env file
load_dotenv()

# Read API key
api_key = os.getenv("OPENAI_API_KEY")

# Initialize client
client = OpenAI(api_key=api_key)

# Example request
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Hello!"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [7]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [8]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1154

In [9]:
documents[1]

{'id': '226a4baf2f',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What’s new in the 2025 edition?',
 'answer': '- Deployment module updated to **FastAPI** (replacing Flask) and new tools.\n- Neural networks taught with **PyTorch** (theory videos in Keras are kept; an additional PyTorch implementation video is provided).\n- Deep learning deployment uses **ONNX Runtime** on AWS Lambda (replacing TensorFlow Lite).'}

In [10]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [11]:
query = 'How do I run Docker on Windows?'
results = index.search(query, num_results=5)

In [12]:
results = index.search(
    query='How do I run Docker on Windows?',
    num_results=5,
    filter_dict={'course': 'mlops-zoomcamp'}
)

In [13]:
results = index.search(
    query='How do I run Docker on Windows?',
    num_results=5,
    boost_dict={'question': 3.0, 'section': 0.5}
)

In [14]:
results[0]['answer']


'You may encounter this error:\n\n```bash\n$ docker run -it ubuntu bash\nthe input device is not a TTY. If you are using mintty, try prefixing the command with \'winpty\'\n```\n\nSolution:\n\n- Use `winpty` before the Docker command:\n  \n  ```bash\n  $ winpty docker run -it ubuntu bash\n  ```\n\n- Alternatively, create an alias:\n  \n  ```bash\n  echo "alias docker=\'winpty docker\'" >> ~/.bashrc\n  ```\n  \n  or\n  \n  ```bash\n  echo "alias docker=\'winpty docker\'" >> ~/.bash_profile\n  ```\n  \nSource: [Stack Overflow](https://stackoverflow.com/a/49965690)'

In [15]:
def search(query, course='mlops-zoomcamp', num_results=5):
    boost_dict = {'question': 3.0, 'section': 0.5}
    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict,
        filter_dict={'course': course}
    )

In [16]:
search('How do I run Docker?', course='mlops-zoomcamp')

[{'id': 'f2458f5a26',
  'course': 'mlops-zoomcamp',
  'section': 'Module 5: Monitoring',
  'question': 'Docker: Docker-Compose deprecated',
  'answer': 'Docker Compose v1 is deprecated from April 2023 onwards. More information on why v2 is better can be found in this blog post:\n\n[New Docker Compose V2 and V1 Deprecation](https://www.docker.com/blog/new-docker-compose-v2-and-v1-deprecation/)'},
 {'id': '418e8948e7',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: How do I start?',
  'answer': 'No matter if you\'re with a \'live\' cohort or following in self-paced mode, start by:\n\n- Reading pins and bookmarks on the course channel to see what things are where.\n\n  <{IMAGE:image_1}>\n\n  <{IMAGE:image_2}>\n\n- Reading the repository (bookmarked in channel) and watching the video lessons (playlist bookmarked in channel).\n\n- If you have questions, search the channel itself first; someone may have already asked and gotten a solutio

In [34]:
INSTRUCTIONS = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

In [18]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [19]:
PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

def build_prompt(query, search_results):
    context = build_context(search_results)
    return PROMPT_TEMPLATE.format(question=query, context=context)

In [20]:
query = 'How do I run Docker on Windows?'
search_results = search(query)
prompt = build_prompt(query, search_results)

print(prompt)

QUESTION: How do I run Docker on Windows?

CONTEXT:
Module 5: Monitoring
Q: Docker: Docker-Compose deprecated
A: Docker Compose v1 is deprecated from April 2023 onwards. More information on why v2 is better can be found in this blog post:

[New Docker Compose V2 and V1 Deprecation](https://www.docker.com/blog/new-docker-compose-v2-and-v1-deprecation/)

General Course-Related Questions
Q: Course: How do I start?
A: No matter if you're with a 'live' cohort or following in self-paced mode, start by:

- Reading pins and bookmarks on the course channel to see what things are where.

  <{IMAGE:image_1}>

  <{IMAGE:image_2}>

- Reading the repository (bookmarked in channel) and watching the video lessons (playlist bookmarked in channel).

- If you have questions, search the channel itself first; someone may have already asked and gotten a solution.

- For the most Frequently Asked Questions, refer to this document:

  <{IMAGE:image_3}>

- If you don't want to read/skimmer/search the FAQ docum

In [21]:
search_results

[{'id': 'f2458f5a26',
  'course': 'mlops-zoomcamp',
  'section': 'Module 5: Monitoring',
  'question': 'Docker: Docker-Compose deprecated',
  'answer': 'Docker Compose v1 is deprecated from April 2023 onwards. More information on why v2 is better can be found in this blog post:\n\n[New Docker Compose V2 and V1 Deprecation](https://www.docker.com/blog/new-docker-compose-v2-and-v1-deprecation/)'},
 {'id': '418e8948e7',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: How do I start?',
  'answer': 'No matter if you\'re with a \'live\' cohort or following in self-paced mode, start by:\n\n- Reading pins and bookmarks on the course channel to see what things are where.\n\n  <{IMAGE:image_1}>\n\n  <{IMAGE:image_2}>\n\n- Reading the repository (bookmarked in channel) and watching the video lessons (playlist bookmarked in channel).\n\n- If you have questions, search the channel itself first; someone may have already asked and gotten a solutio

In [22]:
prompt

'QUESTION: How do I run Docker on Windows?\n\nCONTEXT:\nModule 5: Monitoring\nQ: Docker: Docker-Compose deprecated\nA: Docker Compose v1 is deprecated from April 2023 onwards. More information on why v2 is better can be found in this blog post:\n\n[New Docker Compose V2 and V1 Deprecation](https://www.docker.com/blog/new-docker-compose-v2-and-v1-deprecation/)\n\nGeneral Course-Related Questions\nQ: Course: How do I start?\nA: No matter if you\'re with a \'live\' cohort or following in self-paced mode, start by:\n\n- Reading pins and bookmarks on the course channel to see what things are where.\n\n  <{IMAGE:image_1}>\n\n  <{IMAGE:image_2}>\n\n- Reading the repository (bookmarked in channel) and watching the video lessons (playlist bookmarked in channel).\n\n- If you have questions, search the channel itself first; someone may have already asked and gotten a solution.\n\n- For the most Frequently Asked Questions, refer to this document:\n\n  <{IMAGE:image_3}>\n\n- If you don\'t want to r

In [23]:
response = client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)

In [24]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_012c5c3a3731dc9d006a03157fc73881978390b8ec8bc5896a",
  "created_at": 1778587007.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_012c5c3a3731dc9d006a031580d96481978d518494c5b40163",
      "content": [
        {
          "annotations": [],
          "text": "To run Docker on Windows, the recommended way is to install **Docker Desktop**.\n\n### Basic steps\n1. Download and install **Docker Desktop for Windows**.\n2. Make sure **WSL2** is enabled, since Docker Desktop uses it on Windows.\n3. Open **PowerShell** or **Command Prompt** and test it:\n   ```bash\n   docker --version\n   docker run hello-world\n   ```\n\n### If you're using the course setup and want Docker in WSL2 without Docker Desktop\nYou can install Docker directly inside your WSL2 Linux environment:\n\n```bash\ncurl -fsSL https://get.docker.com -o get-docker.sh\nsudo

In [25]:
response.output[0]

ResponseOutputMessage(id='msg_012c5c3a3731dc9d006a031580d96481978d518494c5b40163', content=[ResponseOutputText(annotations=[], text="To run Docker on Windows, the recommended way is to install **Docker Desktop**.\n\n### Basic steps\n1. Download and install **Docker Desktop for Windows**.\n2. Make sure **WSL2** is enabled, since Docker Desktop uses it on Windows.\n3. Open **PowerShell** or **Command Prompt** and test it:\n   ```bash\n   docker --version\n   docker run hello-world\n   ```\n\n### If you're using the course setup and want Docker in WSL2 without Docker Desktop\nYou can install Docker directly inside your WSL2 Linux environment:\n\n```bash\ncurl -fsSL https://get.docker.com -o get-docker.sh\nsudo sh get-docker.sh\nsudo usermod -aG docker $USER\nsudo systemctl enable docker.service\n```\n\nThen test:\n\n```bash\ndocker --version\ndocker compose version\ndocker run hello-world\n```\n\nIf Docker doesn’t start automatically after restarting WSL, you may need to add a small start

In [26]:
response.output[0].content[0].text

"To run Docker on Windows, the recommended way is to install **Docker Desktop**.\n\n### Basic steps\n1. Download and install **Docker Desktop for Windows**.\n2. Make sure **WSL2** is enabled, since Docker Desktop uses it on Windows.\n3. Open **PowerShell** or **Command Prompt** and test it:\n   ```bash\n   docker --version\n   docker run hello-world\n   ```\n\n### If you're using the course setup and want Docker in WSL2 without Docker Desktop\nYou can install Docker directly inside your WSL2 Linux environment:\n\n```bash\ncurl -fsSL https://get.docker.com -o get-docker.sh\nsudo sh get-docker.sh\nsudo usermod -aG docker $USER\nsudo systemctl enable docker.service\n```\n\nThen test:\n\n```bash\ndocker --version\ndocker compose version\ndocker run hello-world\n```\n\nIf Docker doesn’t start automatically after restarting WSL, you may need to add a small startup command to your shell profile to start the Docker service.\n\nIf you want, I can also give you:\n- the **Docker Desktop + WSL2** 

In [27]:
response.output_text

"To run Docker on Windows, the recommended way is to install **Docker Desktop**.\n\n### Basic steps\n1. Download and install **Docker Desktop for Windows**.\n2. Make sure **WSL2** is enabled, since Docker Desktop uses it on Windows.\n3. Open **PowerShell** or **Command Prompt** and test it:\n   ```bash\n   docker --version\n   docker run hello-world\n   ```\n\n### If you're using the course setup and want Docker in WSL2 without Docker Desktop\nYou can install Docker directly inside your WSL2 Linux environment:\n\n```bash\ncurl -fsSL https://get.docker.com -o get-docker.sh\nsudo sh get-docker.sh\nsudo usermod -aG docker $USER\nsudo systemctl enable docker.service\n```\n\nThen test:\n\n```bash\ndocker --version\ndocker compose version\ndocker run hello-world\n```\n\nIf Docker doesn’t start automatically after restarting WSL, you may need to add a small startup command to your shell profile to start the Docker service.\n\nIf you want, I can also give you:\n- the **Docker Desktop + WSL2** 

In [28]:
response.usage

ResponseUsage(input_tokens=1534, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=264, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1798)

In [29]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0023385000000000003

In [33]:
instructions


NameError: name 'instructions' is not defined

In [37]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    input_messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = client.responses.create(
        model=model,
        input=input_messages
    )

    return response.output_text

In [38]:
instructions = "You're a course teaching assistant."
user_prompt = 'How do I run Docker on Windows?'

llm(instructions, user_prompt)

'To run Docker on Windows, the usual way is to install **Docker Desktop**.\n\n### Basic steps\n1. **Check your Windows edition**\n   - Docker Desktop works best on **Windows 10/11 Pro, Enterprise, or Education**\n   - It can also work on **Windows Home** using **WSL 2**\n\n2. **Enable virtualization**\n   - Make sure **virtualization** is enabled in your BIOS/UEFI\n   - On Windows, you may also need to enable:\n     - **WSL 2**\n     - **Virtual Machine Platform**\n\n3. **Install Docker Desktop**\n   - Download it from: https://www.docker.com/products/docker-desktop/\n   - Run the installer and follow the prompts\n\n4. **Start Docker Desktop**\n   - Open Docker Desktop after installation\n   - Wait until it says Docker is running\n\n5. **Test it**\n   Open PowerShell or Command Prompt and run:\n   ```bash\n   docker --version\n   docker run hello-world\n   ```\n   If it works, Docker is set up correctly.\n\n### If you use Windows Home\nYou’ll typically need to:\n- Install WSL 2\n- Set 

In [39]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [40]:
query = 'How do I run Docker on Windows?'
answer = rag(query)
print(answer)

To run Docker on Windows, install Docker in WSL2 without Docker Desktop:

1. Install Docker:
```bash
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh
```

2. Add your user to the Docker group:
```bash
sudo usermod -aG docker $USER
```

3. Enable the Docker service:
```bash
sudo systemctl enable docker.service
```

4. Test it:
```bash
docker --version
docker compose version
docker run hello-world
```

5. If Docker doesn’t start automatically after restarting WSL, add this to your `.profile` or `.zprofile`:
```bash
if grep -q "microsoft" /proc/version > /dev/null 2>&1; then
   if service docker status 2>&1 | grep -q "is not running"; then
      wsl.exe --distribution "${WSL_DISTRO_NAME}" --user root \
      --exec /usr/sbin/service docker start > /dev/null 2>&1
   fi
fi
```


In [43]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

instructions = '''
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
'''.strip()

assistant = RAGBase(
    index=index,
    llm_client=OpenAI(),
    course='mlops-zoomcamp',
    instructions=instructions,
)

answer = assistant.rag('How do I run Docker on Windows?')
print(answer)

To run Docker on Windows without Docker Desktop, install Docker in WSL2:

1. Install Docker:
```bash
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh
```

2. Add your user to the Docker group:
```bash
sudo usermod -aG docker $USER
```

3. Enable the Docker service:
```bash
sudo systemctl enable docker.service
```

4. Test it:
```bash
docker --version
docker compose version
docker run hello-world
```

If Docker doesn’t start automatically after restarting WSL, add this to your `.profile` or `.zprofile`:
```bash
if grep -q "microsoft" /proc/version > /dev/null 2>&1; then
   if service docker status 2>&1 | grep -q "is not running"; then
      wsl.exe --distribution "${WSL_DISTRO_NAME}" --user root \
      --exec /usr/sbin/service docker start > /dev/null 2>&1
   fi
fi
```
